# Кандидатогенерация для поиска услуг Авито (Recall@50)

Ноутбук по шагам запускает всё решение. Сам код лежит в `.py`-файлах рядом (с подробными комментариями),
здесь — порядок запуска и пояснения. Подробное описание подхода — в `README.md`.

**Структура папки:**
```
dataset/            <- исходные данные: train.parquet, benchmark_queries.parquet, benchmark_items.parquet
work/               <- промежуточные файлы (создаются автоматически)
work/emb_model/     <- дообученная модель эмбеддингов (если есть — шаг 2 можно пропустить)
*.py                <- код решения
```
Установка зависимостей: `pip install -r requirements.txt`

## Шаг 1. Подготовка данных
Нормализация текстов (нижний регистр, ё→е, стемминг Snowball), сборка запросов из train,
отложенные сплиты **val** (2 500 запросов, для оценки) и **rank** (15 000 запросов, для обучения ранкера).
Статистики и модели дальше считаются только по остальной части train — утечки меток нет. (~5 мин)

In [ ]:
%run prep.py

## Шаг 2. Дообучение модели эмбеддингов
Open-source модель `intfloat/multilingual-e5-small` дообучается на парах «запрос → выбранное объявление»
(контрастивная функция потерь, in-batch негативы). Нужна для синонимов: «очистка ног» → «Педикюр».

GPU: ~25 мин, на CPU — несколько часов. **Если папка `work/emb_model` уже есть (прислана вместе с проектом) — шаг пропускается.**

In [ ]:
import os
if os.path.exists("work/emb_model/model.safetensors"):
    print("Модель уже есть в work/emb_model — пропускаем дообучение")
else:
    %run train_emb.py

## Шаг 3. Эмбеддинги объявлений и запросов

In [ ]:
%run encode_emb.py

## Шаг 4. Кандидаты и признаки
Для каждого запроса по всему корпусу считаются: BM25 по полям + локационный приор,
P(микрокатегория | запрос) через kNN по запросам train, история выборов, близость эмбеддингов.
Объединение топов источников (~550 кандидатов на запрос) + ~35 признаков для ранжирования.

In [ ]:
%run features.py val

In [ ]:
%run features.py rank

In [ ]:
%run features.py bench

## Шаг 5. Ранжирование LightGBM и оценка
Обучаем LightGBM (lambdarank) на сплите rank, меряем Recall@50 на val.
Ожидаемо: BM25+локация ≈ 0.82, итоговая модель ≈ 0.936.

In [ ]:
%run ranker.py eval

## Шаг 6. Финальный ответ
Обучаем ранкер на rank+val, предсказываем топ-50 для бенчмарка и пишем `answer.csv`.

In [ ]:
%run ranker.py bench

In [ ]:
%run check_answer.py